# Project 4: Hyperparameter Tuning & Validation

**Dataset:** California Housing

**Goal:** Learn why we need validation sets, how cross-validation works, how to use Grid Search, and how to interpret learning curves.

In [ ]:
# ====== Setup ======
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120
DATA_URL = 'https://raw.githubusercontent.com/lucyyuhongxia-gmail/A4_ML_Projects/main/datasets'
def load_data(fn):
    df = pd.read_csv(f'{DATA_URL}/{fn}')
    print(f'Loaded: {fn} -> {df.shape[0]:,} rows x {df.shape[1]} cols')
    return df
print('Setup OK!')
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, learning_curve
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

## 1. The Overfitting Problem

In [ ]:
df = load_data('housing.csv')
X = df.drop('MedHouseVal', axis=1).select_dtypes(include=[np.number])
y = df['MedHouseVal']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train); X_test_s = scaler.transform(X_test)

# Overfit model (barely any regularization)
overfit = Ridge(alpha=0.0001).fit(X_train_s, y_train)
tr = r2_score(y_train, overfit.predict(X_train_s))
te = r2_score(y_test, overfit.predict(X_test_s))
print(f'Overfit model (alpha=0.0001):')
print(f'  Train R²={tr:.4f}, Test R²={te:.4f}, Gap={tr-te:.4f}')
print(f'  ← High gap = OVERFITTING!')

# Regularized model
good = Ridge(alpha=10).fit(X_train_s, y_train)
tr_g = r2_score(y_train, good.predict(X_train_s))
te_g = r2_score(y_test, good.predict(X_test_s))
print(f'\nRegularized (alpha=10):')
print(f'  Train R²={tr_g:.4f}, Test R²={te_g:.4f}, Gap={tr_g-te_g:.4f}')
print(f'  ← Smaller gap = better generalization')

## 2. K-Fold Cross-Validation

In [ ]:
# 5-Fold Cross-Validation: split data into 5 folds
# Each fold takes a turn as validation set
X_all = StandardScaler().fit_transform(X)
scores = cross_val_score(Ridge(alpha=1.0), X_all, y, cv=5, scoring='r2')

print('5-Fold CV R² Scores:')
for i, s in enumerate(scores):
    print(f'  Fold {i+1}: {s:.4f}')
print(f'  Mean:  {scores.mean():.4f}')
print(f'  Std:   {scores.std():.4f}')
print(f'  Range: {scores.min():.4f} - {scores.max():.4f}')

## 3. Grid Search — Finding Best Alpha

In [ ]:
# Grid Search: try all alpha values, pick the best
grid = GridSearchCV(Ridge(), {'alpha': [0.001,0.01,0.1,0.5,1,5,10,50,100,500]},
    cv=5, scoring='r2', return_train_score=True)
grid.fit(X_all, y)

print(f'Best alpha: {grid.best_params_["alpha"]}')
print(f'Best CV R²: {grid.best_score_:.4f}')
print()
results = pd.DataFrame(grid.cv_results_)
cols = ['param_alpha', 'mean_train_score', 'mean_test_score']
print(results[cols].sort_values('mean_test_score', ascending=False).to_string(index=False))

## 4. Learning Curves

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(
    Ridge(alpha=grid.best_params_['alpha']), X_all, y,
    train_sizes=np.linspace(0.1, 1.0, 10), cv=5, scoring='r2')

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(train_sizes, np.mean(train_scores, axis=1), 'o-', label='Training', color='#2e86de')
ax.plot(train_sizes, np.mean(test_scores, axis=1), 'o-', label='CV', color='#e74c3c')
ax.set_xlabel('Training Examples'); ax.set_ylabel('R² Score')
ax.set_title('Learning Curve'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

gap = np.mean(train_scores[-1]) - np.mean(test_scores[-1])
print(f'Final gap: {gap:.4f}')
if gap < 0.02:
    print('✅ Good: no severe overfitting')
else:
    print('⚠️ Overfitting: consider more regularization or more data')

## Check Your Understanding
1. Why do we need a separate validation set? Why can't we tune on the test set?
2. What does a learning curve tell you? If training score is high but CV score is low, what's happening?
3. Grid Search found alpha=1.0 as best. If you used alpha=0.001 (near zero), what would happen?
4. K in K-Fold CV: is 5 always better than 3? What about 10? What's the trade-off?